# Exploratory component-model training

This notebook is a diagnostic training pass over `data/processed/model_training.csv`. It checks the shared dataset, visualises player/team form and weekly FPL snapshots, selects sensible feature groups, and fits deliberately simple baseline models for each strand in `plan.md`.

The models are for exploration only. They are not saved as production artifacts and the notebook does not replace the versioned scoring engine or walk-forward training jobs planned for later phases.

## Scope and modelling choices

The table is at player-fixture grain. Every input used below is either a pre-match fixture/snapshot field or starts with `feature_`; outcomes start with `target_`. The scored split trains on 2022/23–2024/25 and evaluates on 2025/26. The current 2026/27 season is shown for data checks but is excluded from metrics because Vaastav currently contains only a partial season.

The exploratory strands are:

- team goals for/against;
- appearance, starts, 60+ minutes, and minutes;
- player xG, xA, goals, and assists;
- team and player clean sheets;
- defensive contributions;
- goalkeeper saves;
- bonus and BPS;
- yellow cards, red cards, own goals, and penalty misses.

Non-negative count targets use a Poisson-loss gradient boosting baseline; BPS uses squared-error because its source target can be negative. Event targets use a classifier. This is useful for checking signal, coverage, calibration, and output ranges without implying that these are the final model families.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_poisson_deviance,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
root_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in root_candidates if (path / 'plan.md').exists()), Path.cwd())
DATA_PATH = ROOT / 'data' / 'processed' / 'model_training.csv'
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'{DATA_PATH} does not exist. Run: PYTHONPATH=backend/src python -m preprocessing.build_dataset'
    )

all_columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
data = pd.read_csv(DATA_PATH, parse_dates=['kickoff_time'], low_memory=False)
data['kickoff_time'] = pd.to_datetime(data['kickoff_time'], utc=True)
feature_columns = [column for column in data if column.startswith('feature_')]
target_columns = [column for column in data if column.startswith('target_')]
seasons = sorted(data['season'].dropna().unique())
CURRENT_SEASON = seasons[-1]
completed = data.loc[data['season'].ne(CURRENT_SEASON)].copy()
print(f'Loaded {len(data):,} rows and {len(data.columns):,} columns from {DATA_PATH}')
print(f'Seasons: {seasons}; current partial season excluded from scored metrics: {CURRENT_SEASON}')

In [ ]:
key_columns = ['season', 'player_id', 'fixture_id']
assert not data.duplicated(key_columns).any(), 'duplicate player-fixture keys found'
assert not pd.Index(data.columns).duplicated().any(), 'duplicate column names found'
assert feature_columns and target_columns
assert not set(['minutes', 'starts', 'goals_scored', 'assists']).intersection(data.columns)

season_summary = (
    data.groupby('season', as_index=False)
    .agg(
        rows=('player_id', 'size'),
        players=('player_key', 'nunique'),
        fixtures=('fixture_id', 'nunique'),
        first_gameweek=('gameweek', 'min'),
        last_gameweek=('gameweek', 'max'),
    )
)
display(season_summary)

coverage_columns = [
    'target_expected_goals_available',
    'target_expected_assists_available',
    'target_defensive_contribution_available',
]
coverage = data.groupby('season')[coverage_columns].mean().mul(100).round(2)
display(coverage)

important_columns = [
    'feature_price_tenths',
    'feature_ownership_percent',
    'feature_player_minutes_sum_5',
    'feature_player_expected_goals_sum_5',
    'feature_team_expected_goals_for_mean_5',
    'target_minutes',
    'target_total_points',
    'target_defensive_contribution',
]
missingness = data[important_columns].isna().mean().mul(100).sort_values(ascending=False)
display(missingness.rename('missing_percent').to_frame().round(2))

The coverage table is an intentional modelling constraint: xG/xA are available throughout the configured archive, while defensive-contribution data starts in 2025/26. The latter must be filtered by its availability flag and never backfilled with zeros.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=season_summary, x='season', y='rows', ax=axes[0], color='#4472c4')
axes[0].set_title('Player-fixture rows by season')
axes[0].tick_params(axis='x', rotation=35)

availability = data.groupby('season')[coverage_columns].mean().mul(100).reset_index()
availability = availability.melt('season', var_name='target', value_name='coverage_percent')
sns.barplot(data=availability, x='season', y='coverage_percent', hue='target', ax=axes[1])
axes[1].set_title('Target availability by season')
axes[1].set_ylabel('Rows with target available (%)')
axes[1].tick_params(axis='x', rotation=35)
axes[1].legend(title='', fontsize=8)
plt.tight_layout()

In [ ]:
snapshot = (
    data.sort_values(['season', 'gameweek', 'player_key', 'fixture_id'])
    .drop_duplicates(['season', 'gameweek', 'player_key'], keep='last')
)
weekly_price = snapshot.groupby(['season', 'gameweek'], as_index=False).agg(
    median_price=('feature_price_tenths', 'median'),
    p25_price=('feature_price_tenths', lambda values: values.quantile(0.25)),
    p75_price=('feature_price_tenths', lambda values: values.quantile(0.75)),
)
weekly_price[['median_price', 'p25_price', 'p75_price']] = weekly_price[['median_price', 'p25_price', 'p75_price']].div(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=weekly_price, x='gameweek', y='median_price', hue='season', marker='o', ax=axes[0])
axes[0].set_title('Median weekly player price snapshot')
axes[0].set_ylabel('Price (millions)')

snapshot['price_delta'] = snapshot.groupby(['season', 'player_key'])['feature_price_tenths'].diff()
movement = snapshot.groupby('season')['price_delta'].apply(lambda values: values.fillna(0).ne(0).sum()).reset_index(name='players_with_price_change')
sns.barplot(data=movement, x='season', y='players_with_price_change', ax=axes[1], color='#ed7d31')
axes[1].set_title('Non-zero weekly player price changes')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout()

top_movers = (
    snapshot.groupby('player_key')['price_delta'].apply(lambda values: values.abs().sum()).nlargest(10).index
)
display(snapshot[snapshot['player_key'].isin(top_movers)][['season', 'gameweek', 'player_name', 'feature_price_tenths', 'price_delta']].head(20))

In [ ]:
relation_columns = ['feature_player_expected_goals_sum_5', 'feature_player_expected_goals_per90_5', 'target_expected_goals', 'position']
relation = completed[relation_columns].dropna()
relation = relation.sample(min(len(relation), 12000), random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=relation, x='feature_player_expected_goals_sum_5', y='target_expected_goals', hue='position', alpha=0.25, ax=axes[0])
axes[0].set_title('Pre-match rolling xG versus fixture xG')
axes[0].set_xlabel('Prior rolling xG sum (5 fixtures)')

team_form = completed[['feature_team_expected_goals_for_mean_5', 'feature_opponent_expected_goals_for_mean_5', 'is_home']].dropna()
sns.scatterplot(data=team_form.sample(min(len(team_form), 12000), random_state=RANDOM_STATE), x='feature_team_expected_goals_for_mean_5', y='feature_opponent_expected_goals_for_mean_5', hue='is_home', alpha=0.25, ax=axes[1])
axes[1].set_title('Team and opponent pre-match xG form')
axes[1].set_xlabel('Team rolling expected goals')
axes[1].set_ylabel('Opponent rolling expected goals')
plt.tight_layout()

## Walk-forward split and feature selection

The general evaluation uses complete seasons through 2024/25 for training and 2025/26 for testing. Defensive contributions need a narrower chronological split inside 2025/26 because the source field does not exist in earlier seasons. No current-season rows are used for scored metrics.

Feature groups below deliberately use only context columns and `feature_` prefixes. Player/team IDs are not included, preventing an exploratory tree from memorising identities rather than learning football context.

In [ ]:
completed_seasons = sorted(completed['season'].unique())
TRAIN_SEASONS = completed_seasons[:-1]
TEST_SEASON = completed_seasons[-1]
train_mask = data['season'].isin(TRAIN_SEASONS)
test_mask = data['season'].eq(TEST_SEASON)

defensive_rows = completed.loc[completed['target_defensive_contribution_available'].eq(1)]
defensive_times = np.sort(defensive_rows['kickoff_time'].dropna().unique())
defensive_cutoff = defensive_times[max(1, int(len(defensive_times) * 0.75)) - 1]
defensive_train_mask = data.index.isin(defensive_rows.index[defensive_rows['kickoff_time'] < defensive_cutoff])
defensive_test_mask = data.index.isin(defensive_rows.index[defensive_rows['kickoff_time'] >= defensive_cutoff])
print(f'Train seasons: {TRAIN_SEASONS}; test season: {TEST_SEASON}')
print(f'General split: {train_mask.sum():,} train rows, {test_mask.sum():,} test rows')
print(f'Defensive split cutoff: {pd.Timestamp(defensive_cutoff).date()}')

COMMON_FEATURES = [
    'position',
    'team_name',
    'opponent_team_name',
    'is_home',
    'gameweek',
    'feature_price_tenths',
    'feature_price_millions',
    'feature_ownership_percent',
    'feature_transfers_in_percent',
    'feature_transfers_out_percent',
    'feature_transfers_balance_percent',
    'feature_price_change_previous_snapshot',
    'feature_price_change_season',
    'feature_team_position_price_rank',
    'feature_team_position_ownership_rank',
]
COMMON_FEATURES = [column for column in COMMON_FEATURES if column in data]

def feature_set(*prefixes):
    selected = [column for column in feature_columns if column.startswith(prefixes)]
    return list(dict.fromkeys(COMMON_FEATURES + selected))

FEATURE_SETS = {
    'minutes': feature_set('feature_player_minutes', 'feature_player_appeared', 'feature_player_started', 'feature_player_sixty_plus', 'feature_player_history', 'feature_player_season_history', 'feature_player_rest', 'feature_team_players', 'feature_team_starters', 'feature_team_players_sixty', 'feature_opponent_players', 'feature_opponent_starters', 'feature_opponent_players_sixty', 'feature_team_rest', 'feature_opponent_rest', 'feature_team_gameweek'),
    'team_goals': feature_set('feature_team_goals', 'feature_team_expected_goals', 'feature_opponent_goals', 'feature_opponent_expected_goals', 'feature_team_rest', 'feature_opponent_rest'),
    'attack': feature_set('feature_player_goals', 'feature_player_assists', 'feature_player_expected', 'feature_team_expected_goals', 'feature_opponent_expected_goals'),
    'clean_sheets': feature_set('feature_team_goals_against', 'feature_team_clean_sheet', 'feature_team_expected_goals_against', 'feature_opponent_expected_goals', 'feature_player_minutes', 'feature_player_appeared'),
    'defensive': feature_set('feature_player_defensive', 'feature_player_clearances', 'feature_player_recoveries', 'feature_player_tackles', 'feature_team_defensive', 'feature_opponent_expected_goals'),
    'saves': feature_set('feature_player_saves', 'feature_team_expected_goals_against', 'feature_opponent_expected_goals', 'feature_team_goals', 'feature_player_minutes'),
    'bonus': feature_set('feature_player_bonus', 'feature_player_bps', 'feature_player_influence', 'feature_player_threat', 'feature_player_ict', 'feature_team_total_points'),
    'deductions': feature_set('feature_player_yellow', 'feature_player_red', 'feature_player_own_goals', 'feature_player_penalties', 'feature_player_minutes', 'feature_player_appeared'),
}

feature_catalog = pd.DataFrame(
    [{'group': name, 'columns': len(columns), 'sample_columns': ', '.join(columns[:5])} for name, columns in FEATURE_SETS.items()]
)
display(feature_catalog)
assert all(not column.startswith('target_') for columns in FEATURE_SETS.values() for column in columns)

## Baseline model runner

Missing numeric feature values are median-imputed with missingness indicators, and categorical team/position fields are one-hot encoded. The results table includes a constant training-set baseline so a seemingly good metric can be checked against a trivial prior.

In [ ]:
categorical_columns = ['position', 'team_name', 'opponent_team_name']
model_store = {}
results = []

def make_pipeline(columns, task, regression_loss='poisson'):
    categorical = [column for column in categorical_columns if column in columns]
    numeric = [column for column in columns if column not in categorical]
    preprocess = ColumnTransformer(
        transformers=[
            ('numeric', SimpleImputer(strategy='median', add_indicator=True), numeric),
            ('categorical', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
            ]), categorical),
        ],
        sparse_threshold=0,
    )
    if task == 'classification':
        estimator = HistGradientBoostingClassifier(
            max_iter=60, learning_rate=0.08, max_leaf_nodes=15, random_state=RANDOM_STATE
        )
    else:
        estimator = HistGradientBoostingRegressor(
            loss=regression_loss, max_iter=60, learning_rate=0.08, max_leaf_nodes=15, random_state=RANDOM_STATE
        )
    return Pipeline([('preprocess', preprocess), ('model', estimator)])

def safe_metric(function, *args):
    try:
        return float(function(*args))
    except (ValueError, ZeroDivisionError):
        return np.nan

def run_model(name, strand, target, task, columns, train_selector, test_selector, availability=None, position=None):
    train_selector = pd.Series(train_selector, index=data.index).fillna(False)
    test_selector = pd.Series(test_selector, index=data.index).fillna(False)
    eligible = data[target].notna()
    if availability:
        eligible &= data[availability].eq(1)
    if position:
        eligible &= data['position'].eq(position)
    train_rows = train_selector & eligible
    test_rows = test_selector & eligible
    y_train = data.loc[train_rows, target]
    y_test = data.loc[test_rows, target]
    result = {'model': name, 'strand': strand, 'target': target, 'task': task, 'features': len(columns), 'train_rows': int(train_rows.sum()), 'test_rows': int(test_rows.sum())}
    if len(y_train) < 100 or len(y_test) < 20 or (task == 'classification' and y_train.gt(0).nunique() < 2):
        result['status'] = 'insufficient_training_or_class_variation'
        results.append(result)
        return
    if task == 'regression' and y_train.sum() <= 0:
        prediction = np.zeros(len(y_test))
        result.update({
            'mae': safe_metric(mean_absolute_error, y_test, prediction),
            'rmse': float(np.sqrt(mean_squared_error(y_test, prediction))),
            'poisson_deviance': safe_metric(mean_poisson_deviance, y_test, np.maximum(prediction, 1e-9)),
            'r2': safe_metric(r2_score, y_test, prediction),
            'baseline_mae': result['mae'],
            'baseline_rmse': result['rmse'],
            'status': 'constant_zero_training_target',
        })
        results.append(result)
        return
    X_train = data.loc[train_rows, columns]
    X_test = data.loc[test_rows, columns]
    if task == 'classification':
        y_train = y_train.gt(0).astype(int)
        y_test = y_test.gt(0).astype(int)
    regression_loss = 'poisson' if task == 'classification' or y_train.min() >= 0 else 'squared_error'
    model = make_pipeline(columns, task, regression_loss)
    model.fit(X_train, y_train)
    if task == 'classification':
        probability = model.predict_proba(X_test)[:, 1]
        prediction = (probability >= 0.5).astype(int)
        baseline_probability = np.repeat(float(y_train.mean()), len(y_test))
        result.update({
            'accuracy': safe_metric(accuracy_score, y_test, prediction),
            'roc_auc': safe_metric(roc_auc_score, y_test, probability),
            'log_loss': safe_metric(log_loss, y_test, np.column_stack([1 - probability, probability])),
            'brier': safe_metric(brier_score_loss, y_test, probability),
            'baseline_log_loss': safe_metric(log_loss, y_test, np.column_stack([1 - baseline_probability, baseline_probability])),
            'baseline_brier': safe_metric(brier_score_loss, y_test, baseline_probability),
        })
        model_store[name] = {'model': model, 'X_test': X_test, 'y_test': y_test, 'prediction': prediction, 'probability': probability, 'task': task, 'columns': columns}
    else:
        prediction = np.clip(model.predict(X_test), 0, None)
        baseline_prediction = np.repeat(float(y_train.mean()), len(y_test))
        result.update({
            'mae': safe_metric(mean_absolute_error, y_test, prediction),
            'rmse': float(np.sqrt(mean_squared_error(y_test, prediction))),
            'poisson_deviance': safe_metric(mean_poisson_deviance, y_test, np.maximum(prediction, 1e-9)),
            'r2': safe_metric(r2_score, y_test, prediction),
            'baseline_mae': safe_metric(mean_absolute_error, y_test, baseline_prediction),
            'baseline_rmse': float(np.sqrt(mean_squared_error(y_test, baseline_prediction))),
        })
        model_store[name] = {'model': model, 'X_test': X_test, 'y_test': y_test, 'prediction': prediction, 'task': task, 'columns': columns}
    result['status'] = 'fit'
    results.append(result)

In [ ]:
specifications = [
    ('appearance', 'minutes', 'target_appeared', 'classification', FEATURE_SETS['minutes'], train_mask, test_mask, None, None),
    ('starts', 'minutes', 'target_started', 'classification', FEATURE_SETS['minutes'], train_mask, test_mask, None, None),
    ('sixty_plus', 'minutes', 'target_sixty_plus', 'classification', FEATURE_SETS['minutes'], train_mask, test_mask, None, None),
    ('minutes', 'minutes', 'target_minutes', 'regression', FEATURE_SETS['minutes'], train_mask, test_mask, None, None),
    ('team_goals_for', 'team_goals', 'target_team_goals', 'regression', FEATURE_SETS['team_goals'], train_mask, test_mask, None, None),
    ('team_goals_against', 'team_goals', 'target_opponent_goals', 'regression', FEATURE_SETS['team_goals'], train_mask, test_mask, None, None),
    ('player_xg', 'attack', 'target_expected_goals', 'regression', FEATURE_SETS['attack'], train_mask, test_mask, 'target_expected_goals_available', None),
    ('player_xa', 'attack', 'target_expected_assists', 'regression', FEATURE_SETS['attack'], train_mask, test_mask, 'target_expected_assists_available', None),
    ('player_goals', 'attack', 'target_goals_scored', 'regression', FEATURE_SETS['attack'], train_mask, test_mask, None, None),
    ('player_assists', 'attack', 'target_assists', 'regression', FEATURE_SETS['attack'], train_mask, test_mask, None, None),
    ('team_clean_sheet', 'clean_sheets', 'target_team_clean_sheet', 'classification', FEATURE_SETS['clean_sheets'], train_mask, test_mask, None, None),
    ('player_clean_sheet', 'clean_sheets', 'target_clean_sheets', 'classification', FEATURE_SETS['clean_sheets'], train_mask, test_mask, None, None),
    ('defensive_contribution', 'defensive', 'target_defensive_contribution', 'regression', FEATURE_SETS['defensive'], defensive_train_mask, defensive_test_mask, 'target_defensive_contribution_available', None),
    ('goalkeeper_saves', 'saves', 'target_saves', 'regression', FEATURE_SETS['saves'], train_mask, test_mask, None, 'GK'),
    ('bonus', 'bonus_bps', 'target_bonus', 'regression', FEATURE_SETS['bonus'], train_mask, test_mask, None, None),
    ('bps', 'bonus_bps', 'target_bps', 'regression', FEATURE_SETS['bonus'], train_mask, test_mask, None, None),
    ('yellow_cards', 'deductions', 'target_yellow_cards', 'classification', FEATURE_SETS['deductions'], train_mask, test_mask, None, None),
    ('red_cards', 'deductions', 'target_red_cards', 'classification', FEATURE_SETS['deductions'], train_mask, test_mask, None, None),
    ('own_goals', 'deductions', 'target_own_goals', 'classification', FEATURE_SETS['deductions'], train_mask, test_mask, None, None),
    ('penalty_misses', 'deductions', 'target_penalties_missed', 'classification', FEATURE_SETS['deductions'], train_mask, test_mask, None, None),
]
for specification in specifications:
    run_model(*specification)

metrics = pd.DataFrame(results)
metrics = metrics.sort_values(['strand', 'model']).reset_index(drop=True)
display(metrics)

A useful result is not necessarily a high score: the table should be read with coverage, baseline comparison, class balance, and the plots below. Rare deductions can have undefined ROC-AUC when the held-out block contains one class; that is a data property to address in later calibration work, not a reason to silently drop the strand.

In [ ]:
plot_names = ['appearance', 'team_clean_sheet', 'player_xg', 'defensive_contribution']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for axis, name in zip(axes.flat, plot_names):
    stored = model_store.get(name)
    if stored is None or stored.get('model') is None:
        axis.set_visible(False)
        continue
    actual = stored['y_test']
    if stored['task'] == 'classification':
        calibration = pd.DataFrame({'actual': actual.to_numpy(), 'probability': stored['probability']})
        calibration['probability_bin'] = pd.qcut(calibration['probability'], q=10, duplicates='drop')
        grouped = calibration.groupby('probability_bin', observed=True).agg(predicted=('probability', 'mean'), observed=('actual', 'mean'))
        axis.plot([0, 1], [0, 1], '--', color='grey')
        axis.plot(grouped['predicted'], grouped['observed'], marker='o')
        axis.set_xlabel('Predicted probability')
        axis.set_ylabel('Observed rate')
        axis.set_title(f'{name}: calibration')
    else:
        axis.scatter(actual, stored['prediction'], alpha=0.15, s=8)
        limit = max(float(actual.max()), float(stored['prediction'].max()))
        axis.plot([0, limit], [0, limit], '--', color='grey')
        axis.set_xlabel('Observed')
        axis.set_ylabel('Predicted')
        axis.set_title(f'{name}: observed versus predicted')
plt.tight_layout()

In [ ]:
importance_rows = []
for name in ['appearance', 'team_goals_for', 'player_xg', 'defensive_contribution']:
    stored = model_store.get(name)
    if stored is None:
        continue
    sample_size = min(1500, len(stored['X_test']))
    sample_index = stored['X_test'].sample(sample_size, random_state=RANDOM_STATE).index
    X_sample = stored['X_test'].loc[sample_index]
    y_sample = stored['y_test'].loc[sample_index]
    scoring = 'neg_mean_absolute_error' if stored['task'] == 'regression' else 'neg_log_loss'
    permutation = permutation_importance(stored['model'], X_sample, y_sample, n_repeats=2, random_state=RANDOM_STATE, scoring=scoring, n_jobs=1)
    for column, mean, deviation in zip(stored['columns'], permutation.importances_mean, permutation.importances_std):
        importance_rows.append({'model': name, 'feature': column, 'importance': mean, 'std': deviation})

importance = pd.DataFrame(importance_rows)
top_importance = importance.sort_values(['model', 'importance'], ascending=[True, False]).groupby('model', sort=False).head(12)
display(top_importance)

if not top_importance.empty:
    plt.figure(figsize=(12, 8))
    sns.barplot(data=top_importance, x='importance', y='feature', hue='model', dodge=False)
    plt.title('Permutation importance of selected input columns')
    plt.legend(title='Model')
    plt.tight_layout()

In [ ]:
# The scoring engine is intentionally not recreated here; total FPL points remain a derived target.
point_columns = ['target_total_points', 'target_minutes', 'target_goals_scored', 'target_assists', 'target_clean_sheets', 'target_bonus']
point_summary = completed[point_columns].describe().T[['count', 'mean', '50%', 'max']].round(3)
display(point_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.boxplot(data=completed, x='position', y='target_total_points', ax=axes[0], showfliers=False)
axes[0].set_title('Observed FPL points by position')

target_plot = completed[['target_total_points', 'target_minutes', 'position']].dropna()
sns.scatterplot(data=target_plot.sample(min(len(target_plot), 12000), random_state=RANDOM_STATE), x='target_minutes', y='target_total_points', hue='position', alpha=0.2, ax=axes[1])
axes[1].set_title('Observed points versus minutes')
plt.tight_layout()

print('Exploratory conclusions to carry into model development:')
print('- Use the availability flags when training fields introduced after 2024/25.')
print('- Preserve weekly snapshot fields for price, ownership, and transfers; season-end player metadata is not a substitute.')
print('- Compare every component against its constant baseline and inspect calibration before tuning.')
print('- Feed calibrated component forecasts into the future versioned scoring engine; do not train a direct total-points replacement from this notebook.')

## Limitations and next steps

This notebook intentionally does not tune hyperparameters, perform a full rolling-origin backtest, reconcile player goal forecasts to team goals, model joint score probabilities, or implement 2026/27 FPL scoring. Those belong in the production training/scoring work described in `plan.md`. The notebook's role is to make source coverage, leakage boundaries, feature usefulness, model ranges, and obvious data problems visible before that work begins.